<a href="https://colab.research.google.com/github/Ans365332/6may-file-example/blob/main/Dynamic_tool_registry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Dynamic tool registry -->

In [3]:
!pip install -qU langchain langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 7.9 MB/s eta 0:00:00


In [4]:
import getpass
import os
if "GOOGLE_API_KEY" not in os.environ:
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key:")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 0
)

print("LLM ready:", llm.model)

Enter your Gemini API key:··········
LLM ready: gemini-2.5-flash


In [5]:
import json
import time
import threading
from dataclasses import dataclass, field
from typing import Callable, List, Dict, Any, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

from langchain_core.tools import tool as lc_tool

In [6]:
# Dynamic tool registry --->

# New Workflow Registration
# Old Workflow Delete
# Set the workflow on temporary basis --> if enable or disable

In [7]:
@dataclass
class ToolMeta:
  name: str
  func: Callable
  capabilities: List[str]  #eg. ["math","calculation"]
  required_permission: str   # "public" | "user" | "admin"
  description: str
  enabled: bool = True


class ToolRegistry:
  def __init__(self):
    self._tools: Dict[str, ToolMeta] = {}
    self._lock = threading.Lock()

  def register(self,name, func, capabilities, required_permission= "public", description=""):

    with self._lock:
      self._tools[name] = ToolMeta(
          name=name,
          func=func,
          capabilities=capabilities,
          required_permission=required_permission,
          description=description,
      )

    print(f"[registry] registered tool: '{name}' capabilities={capabilities} perm={required_permission}")

  def unregister(self,name):
    with self._lock:
      if name in self._tools:
        del self._tools[name]
        print(f"[registry] unregistered tool: '{name}'")

  def set_enabled(self, name, enabled: bool):
    with self._lock:
      if name in self._tools:
        self._tools[name].enabled = enabled
        print(f"[registry] tool '{name}' enabled={enabled}")


  def get(self, name):
    return self._tools.get(name)

  def list_tools(self):
    return [t for t in self._tools.values() if t.enabled]

  def get_by_capability(self, capability: str):
    return [t for t in self._tools.values() if t.enabled and capability in t.capabilities]

# Global registry instance
registry = ToolRegistry()
print("Dynamic Tool Registry ready. Currently we have 0 tools")


Dynamic Tool Registry ready. Currently we have 0 tools


In [8]:
# Now we will register some workflows --->

In [9]:
# --- Tool implemetations (palin python functions) ---

def weather_tool(city: str):
  fake_db = {
      "jaipur": "32C, clear sky",
      "mumbai": "29C, humid , light rain",
      "delhi": "36C, hazy",
  }

  return f"The weather in {city} is {fake_db.get(city.lower(), '28C , pleasant (default)')}"

def calculator_tool(expression: str):
  """Safe basic math calculator"""
  allowed = "0123456789+-*/(). "
  if not all(ch in allowed for ch in expression):
    return "Error: invalid characters in expression"

  try:
    return f"Result: {eval(expression)}"
  except Exception as e:
    return f"Error: {e}"

def admin_reset_tool(_: str):
  """Only accessable by admin"""
  return "It will return information sensitive."


In [10]:
# ----- Register all tools dynamically into the registry ----

registry.register("weather", weather_tool,
                  capabilities=["weather","realtime_data"],
                  required_permission="public",
                  description="It will return weather information of any city")

registry.register("calculator_tool", calculator_tool,
                  capabilities=["math","calulation"],
                  required_permission="public",
                  description="Only for claculation")

registry.register("admin_reset_tool", admin_reset_tool,
                  capabilities=["system","admin_action"],
                  required_permission="admin",
                  description="Sensitive system reset action (only for admin)")


print("\nTotal registered tools:", len(registry.list_tools()))




[registry] registered tool: 'weather' capabilities=['weather', 'realtime_data'] perm=public
[registry] registered tool: 'calculator_tool' capabilities=['math', 'calulation'] perm=public
[registry] registered tool: 'admin_reset_tool' capabilities=['system', 'admin_action'] perm=admin

Total registered tools: 3
